# Pandas DataFrames — Solved Exercises (Task 17)

Task 17 marks the transition from the Series to the **DataFrame**, Pandas' two-dimensional table. The first half works on a small hand-built 'birds' dataset where every core DataFrame operation gets a turn — construction, inspection, positional and label selection, boolean filtering, missing-value handling, value counts, duplicate detection, and de-duplication. The second half applies the same tools to a full season-by-season IPL match dataset, where text-cleaning, multi-column aggregation, a reusable function, and plotting all come into play.

The birds dataset is a dictionary of four columns (`birds`, `age`, `visits`, `priority`) plus a list of letter labels `a`–`k`. With it you create a DataFrame via `pd.DataFrame(data, index=labels)`, then run `info()` and `describe()` to see dtypes, non-null counts, and summary statistics in one glance. You slice alternating rows with `.iloc[::2]`, pick specific rows and columns with `df[['birds','age']].iloc[[0,2,6]]`, and filter rows by a condition with `df1[df1.visits < 4]`. Missing ages surface through `df1.age.isna()`, and you fill them using each column's **mode** — `fillna(df1.age.mode()[0])` — rather than the mean, since a categorical-ish column is better imputed with its most common value. The section closes on aggregation and hygiene: summing a subset with `df1[df1.birds == 'Cranes'].visits.sum()`, counting each species with `value_counts()`, and removing duplicate rows permanently with `drop_duplicates(inplace=True)` after quantifying them with `duplicated().sum()`.

The IPL half loads `IPL_Matches_2008_2022.csv` and confronts the reality that franchises get renamed: Delhi Daredevils → Delhi Capitals, Kings XI Punjab → Punjab Kings, Rising Pune Supergiants → Rising Pune Supergiant. You fix the labels with a `replace` of a name-mapping dict — while carefully *not* confusing Gujarat Titans with Gujarat Lions, which are genuinely different teams. Then the analysis: a bar chart of the top-5 teams by matches played (remembering that every match appears in **both** `Team1` and `Team2`), the player with the most Player-of-the-Match awards against Mumbai Indians, a reusable `team1_vs_team2()` head-to-head dashboard function, the top-7 cities where Kolkata Knight Riders play, and Mumbai Indians' average winning margin in the 2011 season.

## 0. Setup

The first two cells stash a URL and import `pandas` and `numpy`.

In [ ]:
"https://raw.githubusercontent.com/justmarkham/DAT8/master/data/chipotle.tsv"

In [ ]:
import pandas as pd
import numpy as np

# 1. Basic DataFrame Operations

Everything here revolves around one small DataFrame built from a Python dictionary. Meet the constructor, the inspection tools, and the selection patterns.

## Basic DataFrame

Consider the following Python dictionary data and Python list labels:

```
data = {'birds': ['Cranes', 'Cranes', 'plovers', 'spoonbills', 'spoonbills', 'Cranes', 'plovers', 'Cranes', 'spoonbills', 'spoonbills', 'Cranes'], 
        'age': [3.5, 4, 1.5, np.nan, 6, 3, 5.5, np.nan, 8, 4, 3.5], 'visits': [2, 4, 3, 4, 3, 4, 2, 2, 3, 2, 2], 
        'priority': ['yes', 'yes', 'no', np.nan, 'no', 'no', 'no', 'yes', 'no', 'no','yes']}
```
```
labels = ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k']
```

### `Q-1:` 
i. Create a DataFrame birds from the above dictionary data which has the index labels. 

ii. Display basic information about the dataFrame.

iii. Show Alternate rows of the dataframe.

## 1.1 Building and inspecting the frame (Q-1)

`pd.DataFrame(data=data, index=labels)` constructs a table from a dict of equal-length column lists, attaching the custom index `['a'..'k']`. The dict keys become column names and each list becomes a column — the natural Python way to hand-build a frame.

For Q-1 ii, `df1.info()` prints a structural summary: the class, the number of index entries (11, from `a` to `k`), each column's name, its count of **non-null** values, and its dtype. The birds output shows `birds` (object) and `visits` (int64) with all 11 entries present, but `age` with only 9 non-null — an immediate flag that two ages are missing. `df1.describe()` complements this with numeric summary statistics: count, mean, std, min, quartiles, and max for the numeric columns (`age` and `visits`), which is where you first notice that `age`'s count is lower still. Running `info()` and `describe()` is the standard two-step opening move on any new frame.

For Q-1 iii, alternate rows come from **step slicing** with `df1.iloc[::2]`: `iloc` is positional indexing, and the slice `start:stop:step` with a step of 2 selects every other row (labels `a, c, e, g, i, k`). Remember the distinction that returns constantly in this notebook: `iloc` selects by *position*, `loc` selects by *label*.

In [ ]:
# code here
data = {'birds': ['Cranes', 'Cranes', 'plovers', 'spoonbills', 'spoonbills', 'Cranes', 'plovers', 'Cranes', 'spoonbills', 'spoonbills', 'Cranes'], 
        'age': [3.5, 4, 1.5, np.nan, 6, 3, 5.5, np.nan, 8, 4, 3.5], 'visits': [2, 4, 3, 4, 3, 4, 2, 2, 3, 2, 2], 
        'priority': ['yes', 'yes', 'no', np.nan, 'no', 'no', 'no', 'yes', 'no', 'no','yes']}
labels = ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k']
#1
df1 = pd.DataFrame(data = data, index = labels)

In [ ]:
#2
df1.info()
df1.describe()

<class 'pandas.core.frame.DataFrame'>
Index: 11 entries, a to k
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   birds     11 non-null     object 
 1   age       9 non-null      float64
 2   visits    11 non-null     int64  
 3   priority  10 non-null     object 
dtypes: float64(1), int64(1), object(2)
memory usage: 440.0+ bytes


            age     visits
count  9.000000  11.000000
mean   4.333333   2.818182
std    1.903943   0.873863
min    1.500000   2.000000
25%    3.500000   2.000000
50%    4.000000   3.000000
75%    5.500000   3.500000
max    8.000000   4.000000

In [ ]:
#3
df1.iloc[::2]

        birds  age  visits priority
a      Cranes  3.5       2      yes
c     plovers  1.5       3       no
e  spoonbills  6.0       3       no
g     plovers  5.5       2       no
i  spoonbills  8.0       3       no
k      Cranes  3.5       2      yes

# 2. Selecting Rows and Columns (Q-2)

Two flavours of selection: pick particular rows/columns by position, and filter rows by a logical condition.

### `Q-2:` 
i. Show only rows [1st, 3rd, 7th] from columns ['bird', 'age']

ii. Select rows where the number of visits is less than 4.

## 2.1 Positional picks and boolean filters

Q-2 i asks for rows `[1st, 3rd, 7th]` from columns `['birds', 'age']`. The solution chains two steps: `df1[['birds', 'age']]` selects those two columns (passing a **list** of names yields a DataFrame, not a Series), and `.iloc[[0, 2, 6]]` picks the zero-based positions 1st, 3rd, and 7th — note the conversion from ordinal language to 0-based indices. The result carries the original index labels (`a`, `c`, `g`), and `.iloc` happily accepts a list of integers to fetch discontiguous rows.

Q-2 ii is a **boolean filter**: `df1[df1.visits < 4]` builds a Series of `True`/`False` (`visits` less than four) and passes it inside the brackets to keep only matching rows. The output shows rows `a, c, e, g, h, i, j, k` — every visit count under four, including row `h` where `age` is `NaN`. This is the single most-used Pandas idiom, so internalise how it reads: the mask has the same length as the frame, and rows where it is `True` survive. Combine multiple conditions with `&`/`|` inside parentheses, as the IPL questions later demonstrate.

In [ ]:
# code here
#1
df1[['birds', 'age']].iloc[[0,2,6]]


     birds  age
a   Cranes  3.5
c  plovers  1.5
g  plovers  5.5

In [ ]:
#2
df1[df1.visits<4]

        birds  age  visits priority
a      Cranes  3.5       2      yes
c     plovers  1.5       3       no
e  spoonbills  6.0       3       no
g     plovers  5.5       2       no
h      Cranes  NaN       2      yes
i  spoonbills  8.0       3       no
j  spoonbills  4.0       2       no
k      Cranes  3.5       2      yes

# 3. Missing Values (Q-3)

Real tables are full of holes. Detecting them and imputing them sensibly is a core cleaning skill.

### `Q-3:` 
i. Select all rows with nan values in age and visits column.

ii. Fill nan with respective series mode value.

## 3.1 Finding and filling nulls

Q-3 i selects all rows that have a `NaN` in *either* the `age` or the `visits` column: `df1[df1.age.isna() | df1.visits.isna()]`. `.isna()` (alias `.isnull()`) returns a boolean mask per column, and joining the two with `|` (logical OR) keeps rows missing in either field. In this dataset rows `d` and `h` are returned — both have missing ages; row `d` also has a missing `priority`.

Q-3 ii fills the gaps with each column's **mode** — the most frequently occurring value — via `df1.age.fillna(df1.age.mode()[0], inplace=True)`. The `[0]` is essential: `mode()` returns a *Series* of all tied modes, so you index the first one. Filling with the mode (rather than the mean used for prices in Task 16) is the right call because these columns are not continuous measurements to be averaged; the 'most common value' is the natural stand-in. The `inplace=True` applies the fix to `df1` directly. Two cautions: imputing always distorts the distribution a little, so record what you did and why; and `inplace` is increasingly discouraged in modern Pandas in favour of reassignment (`df1['age'] = df1['age'].fillna(...)`), which is clearer and chains better.

In [ ]:
# code here
#1
df1[df1.age.isna() | df1.visits.isna()]

        birds  age  visits priority
d  spoonbills  NaN       4      NaN
h      Cranes  NaN       2      yes

In [ ]:
#2
df1.age.fillna(df1.age.mode()[0], inplace=True)
df1.visits.fillna(df1.visits.mode()[0], inplace=True)

# 4. Aggregation, Counts, and Duplicates (Q-4)

Four housekeeping questions that together cover summarising a column, counting categories, and cleaning repeated rows.

### `Q-4`
i. Find the total number of visits of the bird Cranes

ii. Find the number of each type of birds in dataframe.

iii. Print no of duplicate rows

iv. Drop Duplicates rows and make this changes permanent. Show dataframe after changes.

## 4.1 Sum, value counts, duplicates

**i — total visits for Cranes.** `df1[df1.birds == 'Cranes'].visits.sum()` filters to Cranes rows and sums their `visits`, giving **14**. Filter → select column → aggregate is the canonical 'per-group simple statistic' recipe.

**ii — number of each bird type.** `df1.birds.value_counts()` returns the frequency of each distinct species: Cranes 5, spoonbills 4, plovers 2. `value_counts()` is the fastest way to profile a categorical column, and its output is already a Series sorted by count descending — ready to plot.

**iii — count duplicate rows.** `df1.duplicated().sum()` reports **2**: `duplicated()` marks every row that is a repeat of an earlier row, and summing the boolean Series counts them. Crucially, `duplicated()`/`drop_duplicates()` consider **all columns** by default, so two rows identical in every field are duplicates; pass `subset=['birds', ...]` to define duplication by a key.

**iv — drop duplicates permanently.** `df1.drop_duplicates(inplace=True)` removes the repeated rows from `df1` itself. After this the frame is shorter and `df1.duplicated().sum()` would return 0. `keep='first'` (the default) retains the first occurrence; `keep='last'` or `keep=False` change which rows survive — the latter removes *all* members of a duplicate set.

In [ ]:
# code here
#1
df1[df1.birds == "Cranes"].visits.sum()

14

In [ ]:
#2
df1.birds.value_counts()

Cranes        5
spoonbills    4
plovers       2
Name: birds, dtype: int64

In [ ]:
#3
df1.duplicated().sum()

2

In [ ]:
#4
df1.drop_duplicates(inplace=True)

# 5. IPL Match Data (Q-5 to Q-10)

The second dataset is `IPL_Matches_2008_2022.csv`, one row per match. Columns of interest include `Team1`, `Team2`, `WinningTeam`, `Player_of_Match`, `City`, `Season`, and `Margin`.

## Question on IPL Data

IPL Data Link :  https://drive.google.com/file/d/1yKVUuexl6lIKuFQy7uIPgDgXhJ0L4SIg/view?usp=share_link

https://www.kaggle.com/datasets/vora1011/ipl-2008-to-2021-all-match-dataset?select=IPL_Matches_2008_2022.csv

Download ipl matches 2008-2022 file.

## 5.1 Renaming changed franchises (Q-5)

Franchise names evolve across seasons, and analysis must treat old and new names as the same team. The mapping given is:
- `Delhi Daredevils` → `Delhi Capitals`
- `Kings XI Punjab` → `Punjab Kings`
- `Rising Pune Supergiants` → `Rising Pune Supergiant`

The clean solution builds a dict `changed_name` and calls `data.replace(changed_name.keys(), changed_name.values(), inplace=True)`. Passing the keys and values as parallel iterables replaces every occurrence across the whole frame — team names appear in `Team1`, `Team2`, and `WinningTeam`, so a frame-wide replace is exactly right.

Two important cautions. First, the explicit warning that **Gujarat Titans and Gujarat Lions are different teams** — a naive 'Gujarat' shortcut would wrongly merge them, so always map *exact* names. Second, `replace` with `inplace=True` mutates the original frame; if you still need the raw values, work on a copy or assign the result instead. In production you might prefer a `Series.replace` on the relevant columns or `map` with a dict, but the frame-wide form is compact and idiomatic for a small alias list.

### `Q-5:` In IPL matches dataset some teams name has changed. 
You will have to consider them as same.
```
'Delhi Capitals' formerly as 'Delhi Daredevils' 
'Punjab Kings' formerly as 'Kings XI Punjab'
'Rising Pune Supergiant' formerly as 'Rising Pune Supergiants'
```
You need to make changes accordingly. Consider current name for each teams.

Be careful Gujrat Titans and Gujrat Lions are different teams. 

In [ ]:
# code here
data = pd.read_csv('IPL_Matches_2008_2022.csv')
changed_name = {'Delhi Daredevils':'Delhi Capitals', 
                'Kings XI Punjab':'Punjab Kings',
                'Rising Pune Supergiants':'Rising Pune Supergiant'}
data.replace(changed_name.keys(), changed_name.values(),inplace=True )

## 5.2 Top teams by matches played (Q-6)

The trap here is that every match stores two teams, in `Team1` and `Team2`. Counting only one column would halve the truth. The solution adds the two `value_counts()` Series: `data['Team1'].value_counts() + data['Team2'].value_counts()`. Pandas aligns the addition by team name, so each franchise's appearances in either slot are summed correctly (any team missing from one Series contributes zero). Sorting descending and taking `.head()` gives the top five, and `.plot(kind='bar')` renders the bar chart the question asks for.

This 'combine the two symmetric columns, then count' pattern is broadly useful — think home/away goals, sender/receiver messages, or pairwise interactions. Watch the edge cases: `.head()` with no argument defaults to five; the plot's axes labels come automatically from the Series index, though a `.sort_values(ascending=False)` before plotting ensures the bars are ordered; and if a franchise never appears in `Team2`, alignment still handles it. The chart immediately reveals Mumbai Indians, Chennai Super Kings, and Royal Challengers Bangalore as the most-played teams.

### `Q-6` Write a code which can display the bar chart of top 5 teams who have played maximum number of matches in the IPL.
>  Hint: Be careful the data is divided in 2 different cols(Team 1 and Team 2)


In [ ]:
# code here
# Considering both team slots
(data['Team1'].value_counts()+data["Team2"].value_counts()).sort_values(ascending=False).head().plot(kind='bar')

<AxesSubplot:>

<Figure size 640x480 with 1 Axes>

## 5.3 Player-of-the-Match against Mumbai Indians (Q-7)

The question assumes the award goes to a player on the winning team, so first isolate *matches involving* Mumbai Indians, then keep only those **won by the other side** — i.e. MI lost. The expression builds two masks:
```python
m1 = (data.Team1 == 'Mumbai Indians') | (data.Team2 == 'Mumbai Indians')
m2 = data.WinningTeam != 'Mumbai Indians'
data[m1 & m2].Player_of_Match.value_counts().head(1)
```
`m1` selects matches where MI played in either slot; `m2` removes the ones MI won; the `&` keeps games MI played and lost. `value_counts().head(1)` then returns the single most awarded player — **SPD Smith with 4 awards** in this data.

Note the parentheses around each mask: comparisons bind tighter than `&`/`|`, so without them the expression would be misparsed. Also confirm the assumption when you can — the question itself acknowledges it is 'true in most cases', which is exactly the kind of documented caveat a rigorous analyst writes down rather than hides. This mask-combination pattern underlies every multi-condition query in the notebook and the next question packages it into a function.

### `Q-7:` Player who got Most no. of player of the match award playing against Mumbai Indians.
> Just for this question assume player of the match award is given to players from winning team. Although this is true in most of the cases.


In [ ]:
# code here
m1 = (data.Team1 == "Mumbai Indians") | (data.Team2 == "Mumbai Indians")
m2 = data.WinningTeam != "Mumbai Indians"
data[m1 & m2].Player_of_Match.value_counts().head(1)

SPD Smith    4
Name: Player_of_Match, dtype: int64

## 5.4 Head-to-head dashboard function (Q-8)

The task is to factor the selection logic into a reusable function. `team1_vs_team2(t1, t2)` builds one mask per team (`Team1 == t1 or Team2 == t1`) and combines them with `&` to isolate matches where **both** teams appeared — which in a two-team sport means matches *between* them. It then prints `df1.WinningTeam.value_counts()` (the win-loss record) and `df1.Player_of_Match.value_counts().head(1)` (the most-awarded player in the rivalry).

Called with `team1_vs_team2('Kolkata Knight Riders', 'Chennai Super Kings')`, it reports CSK 17 wins to KKR's 9, with `RA Jadeja` the most frequent Player-of-the-Match (3 awards). The value of wrapping the logic in a function is reuse: you can now query any pairing — RCB vs Mumbai, Gujarat vs Rajasthan — without rewriting the masks.

Improvements worth noticing: the returned object could include the city or season split; the print-based function could `return` a dict or DataFrame for programmatic use; and one should validate that the inputs are real team names (after the Q-5 renames). This is a miniature 'analysis as a function' exercise — the first step toward turning ad-hoc queries into reusable tooling.

### `Q-8:` Team1 vs Team2 Dashbord
Create a function which will take two string(name of two teams) as input. Show win Loss record between them and player getting most player of the match award in matches between these two teams.
```
team1_vs_team2('Kolkata Knight Riders','Chennai Super Kings')
```

In [ ]:
# code here
def team1_vs_team2(t1, t2):
    m1 = (data.Team1 == t1) | (data.Team2 == t1)  # Filter for matches where only t1 played
    m2 = (data.Team1 == t2) | (data.Team2 == t2)  # Filter for matches where only t1 played
    df1 = data[m1 & m2]
    print(df1.WinningTeam.value_counts())
    print(df1.Player_of_Match.value_counts().head(1))

team1_vs_team2('Kolkata Knight Riders','Chennai Super Kings')

Chennai Super Kings      17
Kolkata Knight Riders     9
Name: WinningTeam, dtype: int64
RA Jadeja    3
Name: Player_of_Match, dtype: int64


## 5.5 Top cities for Kolkata Knight Riders (Q-9)

The question asks for the seven cities where KKR matches are played most often, drawn as a bar chart. The mask is the now-familiar either-slot filter — `(data.Team1 == 'Kolkata Knight Riders') | (data.Team2 == 'Kolkata Knight Riders')` — then `.City.value_counts().head(7).plot(kind='bar')`. `value_counts()` ranks the cities, `head(7)` keeps the top seven, and the plotting accessor draws them.

The expected result is dominated by Kolkata (Eden Gardens), followed by venues that hosted KKR playoff or neutral matches such as Mumbai, Bangalore, and Delhi. Two refinements to consider: missing `City` values (some matches are recorded with the city blank or with a venue name) should be checked with `.isna().sum()` and optionally dropped; and if you wanted the count normalised or the bars horizontal, `kind='barh'` reads long city names more comfortably. Behind the simplicity is the standard 'filter to a subset, profile a categorical column, plot the top-k' visualisation pattern, applied this time to geography.

### `Q-9:` Find out the top 7 cities where the matches of Kolkata Knight Riders are played frequently and plot the result as bar chart. 

*`.plot(kind = "bar")` can help you to plot the bar chart. Also you can learn more about this method from [here](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.plot.html)*



In [ ]:
# code here
data[(data.Team1 == "Kolkata Knight Riders") | (data.Team2 == "Kolkata Knight Riders")].City.value_counts().head(7).plot(kind='bar')

<AxesSubplot:>

<Figure size 640x480 with 1 Axes>

## 5.6 Average margin for Mumbai Indians in 2011 (Q-10)

Finally, combine three ideas — a team filter, a season filter, and an aggregation. `data[((data.Team1 == 'Mumbai Indians') | (data.Team2 == 'Mumbai Indians')) & (data.Season == '2011')].Margin.mean()` keeps matches MI played **in the 2011 season**, then averages the `Margin` column, returning **19.25**. Reading it carefully: the inner parentheses isolate the either-slot team mask, the `&` intersects it with the season condition, and the surrounding `data[...]` filters the frame. The trailing `.Margin.mean()` selects one numeric column and reduces it.

A few production notes. `Season` may be stored as a string (`'2011'`) or an integer (`2011`) depending on the CSV, so match the literal type — comparing a string column to an integer silently yields all-`False`. `Margin`'s meaning (runs vs wickets) can vary between rows, so averaging blindly may mix units; the question implies run margins, but a careful analyst would filter or inspect first. And when several conditions pile up, assigning each to a named mask variable (as in Q-8) keeps the expression readable. This final query is a neat summary of the whole notebook: filter by entity, filter by context, aggregate a measure.

### `Q-10:` Find out the average margin for the team Mumbai Indians for only the session 2011.

In [ ]:
# code here
data[((data.Team1 == "Mumbai Indians") | (data.Team2 == "Mumbai Indians")) & (data.Season == "2011")].Margin.mean()

19.25

## Summary

Task 17 covered the full DataFrame workflow on two datasets. On the hand-built birds frame you constructed with `pd.DataFrame(data, index=labels)`, inspected with `info()`/`describe()`, took alternate rows using `iloc[::2]`, selected rows and columns positionally with double brackets + `iloc`, filtered by condition (`df1[df1.visits < 4]`), found nulls with `isna()`, and filled them with each column's mode (`fillna(df1.age.mode()[0], inplace=True)`). You summed a filtered subset, profiled species with `value_counts()`, counted duplicates, and dropped them permanently with `drop_duplicates(inplace=True)`. On IPL matches, you renamed franchises via a `replace` mapping (mindful that Gujarat Titans ≠ Gujarat Lions), summed `Team1` and `Team2` counts to chart the top-5 teams, isolated lost matches and found the top Player-of-the-Match against Mumbai Indians, packaged the head-to-head logic into `team1_vs_team2(t1, t2)`, charted KKR's top-7 cities, and averaged Mumbai Indians' 2011 margin (19.25). Core patterns throughout: select, filter, clean, count, aggregate, plot.